# Cross-encoder evidence retriever (FinQANet-style stage 1)

**Research question.** Can a cross-encoder trained on train `gold_inds` retrieve *all* gold facts more
often than train-fitted TF-IDF? Notebook 09 measured TF-IDF all-gold recall@3 at 53.8% on train
(in-sample), and 28.5% for multi-fact records. The generator in Notebook 11 can only use facts that
are retrieved.

**Protocol.** Settings come from `configs/retriever_bert_base.toml`, following the official FinQA
retriever where stated. Train is used for fitting. Dev is used for checkpoint selection and reporting.
The official test split is not loaded. Prediction inputs are built only from `id`, `question`, `table`,
`pre_text` and `post_text`; dev `gold_inds` are read only after scoring, to compute recall.

**Runtime.** This notebook needs a CUDA GPU: follow `docs/gpu-setup.md` first. Setting
`FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check on a few records with a tiny random model. Smoke
outputs go to separate paths and are not results.

In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.evaluation import gold_recall_at_k
from finevidence.retrieval import (
    build_ranking_pairs,
    build_training_pairs,
    load_retriever_config,
    rank_keys,
    sample_epoch_pairs,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
config = load_retriever_config(PROJECT_ROOT / "configs" / "retriever_bert_base.toml")

pretrained_name = (
    "hf-internal-testing/tiny-random-BertModel" if SMOKE_TEST else config.pretrained_name
)
epochs = 1 if SMOKE_TEST else config.epochs
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(str(config.model_dir) + suffix)
model_dir = model_dir if model_dir.is_absolute() else PROJECT_ROOT / model_dir
predictions_path = PROJECT_ROOT / config.predictions_path
if SMOKE_TEST:
    predictions_path = predictions_path.with_name(predictions_path.stem + "_smoke.jsonl")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "pretrained_name": pretrained_name,
    "epochs": epochs,
    "seed": config.seed,
}
print(json.dumps(environment, indent=2))

{
  "commit": "7028afca544a17c9ba05f488e795cb66aba06b4a",
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "transformers": "5.16.1",
  "device": "Tesla T4",
  "smoke_test": false,
  "pretrained_name": "bert-base-uncased",
  "epochs": 5,
  "seed": 42
}


## 1. Verified data and retriever pairs

The acquisition helper downloads train and dev only if they are missing, and verifies both against the
pinned SHA-256 values. Positives and negatives use the same header-paired rendering of the raw context
(see `finevidence.retrieval.pairs`).

In [2]:
raw_dir = PROJECT_ROOT / config.dataset_root
print(download_finqa_splits(raw_dir, (config.train_split, config.evaluation_split)))
train = load_finqa_split(raw_dir / f"{config.train_split}.json")
dev = load_finqa_split(raw_dir / f"{config.evaluation_split}.json")
if SMOKE_TEST:
    train, dev = train[:40], dev[:20]

positives, negative_pool, unmatched_gold = build_training_pairs(
    train, include_header_row=config.include_header_row
)
dev_pairs = {
    record["id"]: build_ranking_pairs(record, include_header_row=config.include_header_row)
    for record in dev
}
print(
    {
        "train_records": len(train),
        "dev_records": len(dev),
        "test_loaded": False,
        "train_positives": len(positives),
        "train_negative_pool": len(negative_pool),
        "pairs_per_epoch": len(positives) * (1 + config.negatives_per_positive),
        "dev_pairs": sum(len(pairs) for pairs in dev_pairs.values()),
        "unmatched_train_gold_keys": dict(unmatched_gold),
    }
)
display(
    pd.DataFrame([vars(pair) for pair in (*positives[:3], *negative_pool[:3])])[
        ["record_id", "key", "label", "question", "text"]
    ]
)

{'train': 'existing; verified, not overwritten', 'dev': 'existing; verified, not overwritten'}


{'train_records': 6251, 'dev_records': 883, 'test_loaded': False, 'train_positives': 10691, 'train_negative_pool': 180663, 'pairs_per_epoch': 42764, 'dev_pairs': 26834, 'unmatched_train_gold_keys': {'text_-1': 1}}


,record_id,key,label,question,text
0,ADI/2009/page_49.pdf-1,text_1,1,what is the the interest expense in 2009?,"if libor changes by 100 basis points , our ann..."
1,ABMD/2012/page_75.pdf-1,table_2,1,"during the 2012 year , did the equity awards i...",row: granted | number of shares ( in thousands...
2,ABMD/2012/page_75.pdf-1,text_15,1,"during the 2012 year , did the equity awards i...","during the year ended march 31 , 2012 , the co..."
3,ADI/2009/page_49.pdf-1,table_0,0,what is the the interest expense in 2009?,header: | october 31 2009 | november 1 2008
4,ADI/2009/page_49.pdf-1,table_1,0,what is the the interest expense in 2009?,row: fair value of forward exchange contracts ...
5,ADI/2009/page_49.pdf-1,table_2,0,what is the the interest expense in 2009?,row: fair value of forward exchange contracts ...


## 2. TF-IDF reference on dev

This is the lexical reference on the same rendering. The vocabulary is fitted on train questions and
train segment texts only. Dev gold keys are read only after the rankings are fixed.

In [3]:
def recall_table(rankings: dict[str, list[str]], records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        gold = [key for key in record["qa"]["gold_inds"] if key in rankings[record["id"]]]
        if not gold:
            continue
        for k in config.recall_k:
            recall = gold_recall_at_k(rankings[record["id"]], gold, k)
            rows.append(
                {
                    "k": k,
                    "any_gold": recall.any_gold,
                    "all_gold": recall.all_gold,
                    "multi_fact": recall.gold_count > 1,
                }
            )
    frame = pd.DataFrame(rows)
    table = frame.groupby("k")[["any_gold", "all_gold"]].mean()
    table["all_gold_multi_fact"] = frame[frame.multi_fact].groupby("k")["all_gold"].mean()
    return table.round(4)


vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(
    [pair.text for pair in (*positives, *negative_pool)]
    + [record["qa"]["question"] for record in train]
)
tfidf_rankings = {}
for record_id, pairs in dev_pairs.items():
    scores = (
        vectorizer.transform([p.text for p in pairs]) @ vectorizer.transform([pairs[0].question]).T
    )
    tfidf_rankings[record_id] = rank_keys([p.key for p in pairs], scores.toarray().ravel().tolist())
tfidf_recall = recall_table(tfidf_rankings, dev)
display(tfidf_recall)

,any_gold,all_gold,all_gold_multi_fact
k,,,
1,0.5878,0.2922,0.0000
3,0.8256,0.5345,0.2719
5,0.9015,0.6602,0.4497
10,0.9604,0.8414,0.7452


## 3. Train the cross-encoder

`AutoModelForSequenceClassification` with two labels scores `(question, segment)` pairs; only the
segment side is truncated. Each epoch resamples negatives with a seed derived from the config seed
and the epoch. The checkpoint with the best dev all-gold recall at `top_n_for_generator` is kept.
Mixed precision is used on CUDA.

In [4]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
model = AutoModelForSequenceClassification.from_pretrained(pretrained_name, num_labels=2).to(device)


def encode(pairs):
    return tokenizer(
        [pair.question for pair in pairs],
        [pair.text for pair in pairs],
        truncation="only_second",
        max_length=config.max_seq_length,
        padding=True,
        return_tensors="pt",
    ).to(device)


@torch.no_grad()
def rank_dev(label: str) -> tuple[dict[str, list[str]], dict[str, list[float]]]:
    model.eval()
    rankings, scores_by_record = {}, {}
    started = time.time()
    report_every = max(1, len(dev_pairs) // 4)
    for done, (record_id, pairs) in enumerate(dev_pairs.items(), start=1):
        scores = []
        for start in range(0, len(pairs), config.batch_size * 4):
            batch = pairs[start : start + config.batch_size * 4]
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                logits = model(**encode(batch)).logits
            scores.extend(torch.softmax(logits.float(), dim=-1)[:, 1].tolist())
        rankings[record_id] = rank_keys([p.key for p in pairs], scores)
        scores_by_record[record_id] = scores
        if done % report_every == 0 or done == len(dev_pairs):
            print(
                f"  {label}: ranked {done}/{len(dev_pairs)} dev records ({clock(time.time() - started)})",
                flush=True,
            )
    return rankings, scores_by_record


steps_per_epoch = -(-len(positives) * (1 + config.negatives_per_positive) // config.batch_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(config.warmup_ratio * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
selection_k = config.top_n_for_generator
history, best_recall = [], -1.0
model_dir.mkdir(parents=True, exist_ok=True)
report_every = max(1, steps_per_epoch // 20)
run_started = time.time()
print(
    f"Training {epochs} epoch(s) x {steps_per_epoch} steps, batch {config.batch_size}, on {environment['device']}",
    flush=True,
)

for epoch in range(epochs):
    model.train()
    epoch_pairs = sample_epoch_pairs(
        positives,
        negative_pool,
        negatives_per_positive=config.negatives_per_positive,
        seed=config.seed,
        epoch=epoch,
    )
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(epoch_pairs), config.batch_size), start=1):
        batch = epoch_pairs[start : start + config.batch_size]
        labels = torch.tensor([pair.label for pair in batch], device=device)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            loss = model(**encode(batch), labels=labels).loss
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            remaining = elapsed / step * (steps_per_epoch - step)
            recent = float(np.mean(losses[-report_every:]))
            print(
                f"  step {step}/{steps_per_epoch} ({100 * step / steps_per_epoch:.0f}%)"
                f" | loss {recent:.4f} | elapsed {clock(elapsed)} | epoch time left ~{clock(remaining)}",
                flush=True,
            )
    train_seconds = time.time() - started
    rankings, _ = rank_dev(f"epoch {epoch + 1} dev")
    recall = recall_table(rankings, dev)
    selected = float(recall.loc[selection_k, "all_gold"])
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            f"dev_all_gold_at_{selection_k}": selected,
            f"dev_any_gold_at_{selection_k}": float(recall.loc[selection_k, "any_gold"]),
        }
    )
    improved = selected > best_recall
    if improved:
        best_recall = selected
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    # Written every epoch so an interrupted run keeps its completed epochs.
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    per_epoch = (time.time() - run_started) / (epoch + 1)
    print(
        f"Epoch {epoch + 1}/{epochs} done: {history[-1]}"
        f" | {'new best, checkpoint saved' if improved else 'not better than best'}"
        f" | total elapsed {clock(time.time() - run_started)}"
        f" | ~{clock(per_epoch * (epochs - epoch - 1))} left for {epochs - epoch - 1} epoch(s)",
        flush=True,
    )
display(pd.DataFrame(history))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training 5 epoch(s) x 2673 steps, batch 16, on Tesla T4



Epoch 1/5


  step 133/2673 (5%) | loss 0.6555 | elapsed 0m14s | epoch time left ~4m40s


...

  epoch 5 dev: ranked 660/883 dev records (0m30s)


  epoch 5 dev: ranked 880/883 dev records (0m40s)


  epoch 5 dev: ranked 883/883 dev records (0m40s)


Epoch 5/5 done: {'epoch': 4, 'train_loss': 0.06672446295936768, 'train_seconds': 299.2, 'epoch_seconds_with_dev': 340.0, 'dev_all_gold_at_5': 0.8437, 'dev_any_gold_at_5': 0.9751} | not better than best | total elapsed 28m25s | ~0m00s left for 0 epoch(s)


,epoch,train_loss,train_seconds,epoch_seconds_with_dev,dev_all_gold_at_5,dev_any_gold_at_5
0,0,0.273269,297.6,338.3,0.8539,0.9841
1,1,0.175967,301.3,342.0,0.8686,0.9853
2,2,0.130875,300.0,340.7,0.8698,0.9887
3,3,0.093374,301.7,342.4,0.8630,0.9819
4,4,0.066724,299.2,340.0,0.8437,0.9751


## 4. Selected checkpoint on dev and ranking artifact

The selected checkpoint re-ranks every dev record. The full ranking with scores is written to an ignored
path for Notebook 11. The cross-encoder and TF-IDF are then compared on the same dev records and rendering.

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_dir / "best").to(device)
rankings, scores_by_record = rank_dev("selected checkpoint")
cross_encoder_recall = recall_table(rankings, dev)
comparison = pd.concat({"tfidf": tfidf_recall, "cross_encoder": cross_encoder_recall}, axis=1)
display(comparison)

predictions_path.parent.mkdir(parents=True, exist_ok=True)
with predictions_path.open("w", encoding="utf-8") as handle:
    for record_id, pairs in dev_pairs.items():
        score_by_key = dict(zip([p.key for p in pairs], scores_by_record[record_id], strict=True))
        ranked = rankings[record_id]
        handle.write(
            json.dumps(
                {
                    "id": record_id,
                    "ranked_keys": ranked,
                    "scores": [round(score_by_key[key], 6) for key in ranked],
                    "checkpoint": str(model_dir / "best"),
                    "commit": commit,
                }
            )
            + "\n"
        )
print(
    {"rankings": str(predictions_path), "records": len(rankings), "best_epoch_metric": best_recall}
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  selected checkpoint: ranked 220/883 dev records (0m10s)


  selected checkpoint: ranked 440/883 dev records (0m20s)


  selected checkpoint: ranked 660/883 dev records (0m30s)


  selected checkpoint: ranked 880/883 dev records (0m40s)


  selected checkpoint: ranked 883/883 dev records (0m40s)


tfidf                              cross_encoder           \
   any_gold all_gold all_gold_multi_fact      any_gold all_gold   
k                                                                 
1    0.5878   0.2922              0.0000        0.8947   0.4134   
3    0.8256   0.5345              0.2719        0.9728   0.8075   
5    0.9015   0.6602              0.4497        0.9887   0.8698   
10   0.9604   0.8414              0.7452        0.9943   0.9354   

                        
   all_gold_multi_fact  
k                       
1               0.0000  
3               0.6724  
5               0.7709  
10              0.8865

{'rankings': '/kaggle/working/finevidence/data/processed/retriever_bert_base_dev_rankings.jsonl', 'records': 883, 'best_epoch_metric': 0.8698}


## Interpretation

Run: Kaggle Tesla T4, commit `7028afc`, torch 2.11.0+cu128, transformers 5.16.1, seed 42, 28 min 25 s in total (about 300 s of training and 40 s of dev ranking per epoch). The outputs above are from that run. Re-scoring the saved dev rankings locally reproduced every recall value below to four decimals.

- **The cross-encoder retrieves complete evidence far more often than TF-IDF.** Dev all-gold recall@3 is 0.8075 vs 0.5345 and @5 is 0.8698 vs 0.6602. At k = 5, records missing at least one gold fact fall from about 300 to about 115 of 883.
- **The gain is largest where Notebook 09 found TF-IDF weakest.** For the 467 multi-fact dev records, all-gold@3 rises from 0.2719 to 0.6724 and @5 from 0.4497 to 0.7709. The top-ranked segment is gold for 89.5% of records (TF-IDF: 58.8%).
- **Training length.** Dev all-gold@5 peaked at epoch 3 (0.8698), then fell to 0.8630 and 0.8437 while training loss kept falling (0.273 → 0.067). Longer training appears to overfit, so the epoch-3 checkpoint is kept.
- **Limitations.** The checkpoint was selected on dev and is also reported on dev, so these figures are slightly optimistic. Epochs 2–4 differ by under 0.007, so the bias is small, but unbiased numbers require the held-out M4 evaluation. This is a single seed on one GPU type. Recall measures whether evidence is available, not answer correctness.

**Decision.** Use this checkpoint's dev rankings (top 5, `data/processed/retriever_bert_base_dev_rankings.jsonl`) as the generator input in Notebook 11. The roughly 13% of dev records without complete evidence in the top 5 limit end-to-end accuracy; a gold-facts oracle run will separate that limit from generator errors.